# 08장 보안 실습 — 지속성 위치와 승인 검토


## Goal

자동 시작 위치의 승인·실행 주체·수집 누락을 구분합니다.

[교안과 분석 질문](../../08-system-automation/08-3-persistence-review.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-08-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'persistence.psv': 'mechanism|path|owner|approval|observed_kst\nsystemd|/etc/systemd/system/report-helper.service|root|unknown|2026-09-10T09:05:00+09:00\ncron|/etc/cron.d/backup|root|CHG-100|2026-09-09T18:00:00+09:00\nshell-startup|/home/analyst/.bashrc|analyst|baseline|2026-09-01T10:00:00+09:00\nssh-key|/home/analyst/.ssh/authorized_keys|analyst|KEY-200|2026-09-01T10:00:00+09:00\n', 'service-review.txt': '# Synthetic review excerpt only. Not an installable unit file.\nunit=report-helper.service\nUser=collector\nExecStart=/opt/collector/bin/report\nFragmentPath=/etc/systemd/system/report-helper.service\nDropInPaths=not_collected\nchange_ticket=unknown\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 승인 미확인 1개, 알려진 기록 3개, 설치한 지속성 0개

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 자동 시작 위치의 승인 상태 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4=="unknown" {print $1 "|" $2}' "$COURSE_DATA/persistence.psv" > "$COURSE_OUT/unapproved.psv"
test "$(wc -l < "$COURSE_OUT/unapproved.psv")" -eq 1
cat "$COURSE_OUT/unapproved.psv"


### 2. 서비스 실행 주체와 경로 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -E '^(User|ExecStart|DropInPaths|change_ticket)=' "$COURSE_DATA/service-review.txt" > "$COURSE_OUT/service-context.txt"
grep -Fx 'User=collector' "$COURSE_OUT/service-context.txt"
grep -Fx 'DropInPaths=not_collected' "$COURSE_OUT/service-context.txt"


### 3. 정상 기준선도 함께 남기기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4!="unknown" {print $1 "|" $4}' "$COURSE_DATA/persistence.psv" > "$COURSE_OUT/known.psv"
test "$(wc -l < "$COURSE_OUT/known.psv")" -eq 3
printf 'unknown_approval=1 known_records=3 installed_by_lab=0\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: 자동 시작 구성의 승인과 실행은 같은가

**Red Team 질문:** 재부팅·예약·로그인 시 실행되는 구성의 변경 권한이 적절히 제한되어 있는가? 지속성은 동작을 계속 실행시키려는 목적이며, cron이나 systemd가 설치되어 있다는 사실 자체가 약점은 아닙니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 자동 실행의 연속성에 영향을 줄 수 있는 설정·코드의 통제 권한 검토 |
| Command / Observation | grep으로 User·ExecStart·DropInPaths·change_ticket을 선택. 현재 drop-in 미수집·승인 미확인 |
| System Change / Artifact | unit·timer·crontab·시작 파일·키의 변경. 존재·활성화·실행은 각각 별도 사실 |
| Log prerequisite | 사전 파일 변경 감사·배포 이력, 서비스 Journal·인증 로그. 모든 시작 파일 읽기가 자동 기록되지는 않음 |
| Blue Team Investigation | 등록된 위치→실제 참조 구성→실행 계정→파일 변경 주체→동일 기간 실행 기록 비교 |
| Detection | 승인되지 않은 구성 변화와 후속 실행을 연결. 미수집을 비인가 확정으로 바꾸지 않음 |
| Mitigation | 구성·코드의 쓰기 권한 제한, 변경 승인, 키 관리. 보존 후 승인된 정상 구성으로 복구 |

**반례와 해설:** 정상 백업 cron도 반복 실행되고 모니터링 unit도 부팅 때 시작됩니다. 합성 collector는 실행 계정이 collector이므로 root 지속성이라고 쓸 수 없습니다. DropInPaths=not_collected이면 본 파일만 보고 최종 실행 옵션을 완전히 확인했다는 표현도 틀립니다.

### 조사 위치별 다음 질문

| 위치 | 추가 확인 | 방어 측 검증 |
|---|---|---|
| cron·timer | 어떤 사용자와 시간 조건인가? | 예약과 실제 실행·변경 승인을 비교 |
| Shell 시작 파일 | 어느 셸 모드에서 참조되는가? | source로 실행하지 않고 참조 관계·수집 범위 확인 |
| authorized_keys | 실제 설정의 키 경로와 승인 주체는? | 지문·변경 이력·인증 방식·성공 기록 비교 |
| 동적 링커 | 구성·라이브러리가 배포 기준선과 맞는가? | 파일 신원·변경 이력·수집된 로드 정보 구분 |
| 커널 모듈 | 배포 패키지·서명·허용 목록과 맞는가? | 현재 로드 정보와 부팅 기록 비교. 이름만으로 악성 판정하지 않음 |

**제출 과제:** 승인 미확인 1건에 대해 필요한 자료 두 개와 가능한 정상 사유를 적습니다. “지속성 설치 확인” 대신 현재 확인된 설정·실행·승인 상태를 각각 기술합니다. 실습에서 어떤 구성도 설치하지 않았음을 결과에 남깁니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
